# DS 3100 Case Study: Tennessee Education Data — Wrangling and Joins

**Goal:** Practice data-wrangling operations and learn why joining datasets is sometimes necessary to answer an analytical question.

This notebook intentionally stops **before EDA and visualization**. A separate exercise will use the resulting analytical data for exploratory analysis and `ggplot2` visualizations.

## 0. The datasets

We will work with two tables:

1. **`tenn2018.csv`** — Tennessee 2018 achievement data.
2. **`tn_district_info.csv`** — a small companion lookup table containing one row per district and a few district-level characteristics.

The achievement file contains district/school identifiers, subgroup, subject, current achievement measures, and previous-year measures. The companion file is deliberately simpler so that we can focus on the mechanics and reasoning of a join.

**Download the companion dataset:** [tn_district_info.csv](tn_district_info.csv)

In [ ]:
library(dplyr)

# Read the two datasets
# Update the paths if your files are stored elsewhere.
ten <- read.csv("tenn2018.csv")
district_info <- read.csv("tn_district_info.csv")

## 1. Get to know the achievement data

Before wrangling, identify the **unit of observation** and the variables that will drive the analysis.

In [ ]:
# Dimensions
nrow(ten)
ncol(ten)

# Variable names
names(ten)

# First few observations
head(ten)

### Think about the level of observation

The dataset contains statewide, district-level, and individual-school records. For this case study we want **district-level observations**.

- `district_number = 0`, `school_number = 0` → statewide aggregate
- `district_number > 0`, `school_number = 0` → district aggregate
- `district_number > 0`, `school_number > 0` → individual school

In [ ]:
# Keep district-level aggregates only
analysis <- ten %>%
  filter(
    district_number > 0,
    school_number == 0
  )

head(analysis)

## 2. Define the analysis population

We want to compare achievement across subjects for the **All Students** subgroup.

Keep:

- `subgroup == "All Students"`
- ELA and Math observations

Then keep only the variables needed for the analysis.

In [ ]:
analysis <- analysis %>%
  filter(
    subgroup == "All Students",
    overall_subject %in% c("ELA", "Math")
  ) %>%
  select(
    district_number,
    district_name,
    overall_subject,
    percent_below,
    percent_on_mastered,
    percent_below_previous,
    percent_on_mastered_previous
  )

head(analysis)

## 3. Create variables for change over time

The dataset provides corresponding achievement measures for the current year and the previous/baseline year.

Create two new variables:

- `change_below` = current `percent_below` − previous `percent_below`
- `change_on_mastered` = current `percent_on_mastered` − previous `percent_on_mastered`

Remember that the direction of a favorable change is different for the two measures.

In [ ]:
analysis <- analysis %>%
  mutate(
    change_below = percent_below - percent_below_previous,
    change_on_mastered = percent_on_mastered - percent_on_mastered_previous
  )

head(analysis)

## 4. Handle missing values deliberately

Before summarizing, inspect missingness rather than allowing it to remain invisible.

**Questions:**

- Which variables contain missing values?
- Why might a change variable be missing even when the current-year measure is available?

In [ ]:
colSums(is.na(analysis))

For the next summary, we will let `mean(..., na.rm = TRUE)` ignore missing values **within the calculation**. This is different from filtering rows out of the dataset.

## 5. Group and summarize

Before joining another dataset, we can already answer some questions about achievement by subject.

For each subject, calculate:

- number of non-missing `percent_below` observations;
- mean `percent_below`;
- mean `percent_on_mastered`;
- mean `change_below`.

In [ ]:
subject_summary <- analysis %>%
  group_by(overall_subject) %>%
  summarize(
    n_observations = sum(!is.na(percent_below)),
    mean_percent_below = mean(percent_below, na.rm = TRUE),
    mean_percent_on_mastered = mean(percent_on_mastered, na.rm = TRUE),
    mean_change_below = mean(change_below, na.rm = TRUE)
  )

subject_summary

## 6. Why do we need another dataset?

Suppose we now want to ask:

> **Do achievement outcomes differ across districts of different sizes?**

The achievement table does not contain the district-size classification we need. We therefore need information from another table.

The companion `tn_district_info.csv` is a deliberately small lookup table with **one row per district**.

In [ ]:
head(district_info)

# Check that the lookup table has one row per district
nrow(district_info)
sum(duplicated(district_info[c("district_number", "district_name")]))

## 7. Identify the join key

The two tables share:

- `district_number`
- `district_name`

We will use the **pair of columns together** as a composite key.

A match requires both values to agree.

We will use a **left join** because we want to preserve every observation in our achievement analysis and attach district information where a match exists.

In [ ]:
joined <- analysis %>%
  left_join(
    district_info,
    by = c("district_number", "district_name")
  )

head(joined)

## 8. Validate the join

A join can succeed syntactically and still be analytically wrong. Check what happened.

Ask:

1. Did the number of observations change?
2. Did every district receive a district-size value?
3. Did the join create unexpected missing values?

In [ ]:
# Compare row counts
nrow(analysis)
nrow(joined)

# Check unmatched district information
sum(is.na(joined$district_size))

# Inspect the district-size categories
table(joined$district_size, useNA = "ifany")

## 9. Post-join wrangling

Now that the district information is available, we can use it in another grouped summary.

Calculate the mean `percent_below` for each district-size category.

In [ ]:
size_summary <- joined %>%
  group_by(district_size) %>%
  summarize(
    n_observations = sum(!is.na(percent_below)),
    mean_percent_below = mean(percent_below, na.rm = TRUE)
  )

size_summary

## 10. Final wrangling challenge

Use the joined dataset to answer:

> **For ELA only, how does the average `percent_on_mastered` differ across district-size categories?**

Then create a second summary that compares the average `change_below` across the same categories.

### Before moving on

Be ready to explain:

- why we filtered to district-level observations;
- why `district_number` and `district_name` form a useful composite key;
- why we chose a left join; and
- what `na.rm = TRUE` does when calculating a summary.

In [ ]:
# Your turn: complete these summaries

ela_size_summary <- joined %>%
  filter(overall_subject == "ELA") %>%
  group_by(district_size) %>%
  summarize(
    mean_percent_on_mastered = mean(percent_on_mastered, na.rm = TRUE),
    mean_change_below = mean(change_below, na.rm = TRUE)
  )

ela_size_summary

## 11. Wrap-up

At this point we have moved from raw tables to an analysis-ready dataset by:

**filtering → selecting → creating variables → handling missingness → grouping → summarizing → joining → validating the join → summarizing again**

The resulting `joined` data are ready for a separate **EDA and visualization** exercise using `ggplot2`.

In [ ]:
# Optional: save the joined analysis data for the next exercise
write.csv(joined, "tn_joined_analysis.csv", row.names = FALSE)